# TARDIS - Dataset Cleaning Notebook

This notebook is part of the TARDIS project at Epitech. It focuses on cleaning the raw dataset of train traffic to prepare it for analysis and modeling. The process includes:

- Handling duplicates and missing values
- Validating data types and formats
- Ensuring consistency in delay-related metrics
- Cleaning textual anomalies
- Performing basic feature engineering
- Aggregating data if necessary

Each section is annotated and tested to ensure compatibility with validation scripts.

## Step 1 — Load and Inspect the Raw Dataset

We begin by loading the dataset and performing a preliminary inspection:
- Column names
- Row count
- Data types
- Quick sample

In [ ]:
import pandas as pd

df = pd.read_csv("dataset.csv", delimiter=";")

print(f"Initial shape: {df.shape}")
df.head()

In [ ]:
# Quick look at column types and missing values
df.info()
df.isnull().sum().sort_values(ascending=False)

## Step 2 — Basic Data Quality Checks

We now perform checks to ensure the integrity of the dataset:
- Remove duplicates
- Identify missing values
- Validate the date format

In [ ]:
# Check for duplicate rows
num_duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {num_duplicates}")

# Remove duplicates
df = df.drop_duplicates()
print(f"Shape after removing duplicates: {df.shape}")

In [ ]:
# Show missing value counts
missing_counts = df.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

if missing_counts.empty:
    print("✅ No missing values found.")
else:
    print("⚠️ Missing values:")
    print(missing_counts)

In [ ]:
# Check if 'date' column exists and convert to datetime
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    num_invalid_dates = df["Date"].isnull().sum()
    print(f"Number of invalid dates after conversion: {num_invalid_dates}")
else:
    print("❌ 'date' column not found.")

## Step 3 — Numeric Validation and Negative Value Filtering

We identify numeric columns, convert them as needed, and ensure that columns representing counts or durations do not contain negative values.

In [ ]:
# Columns where values should be numeric and non-negative
positive_numeric_cols = [
    "Average journey time",
    "Number of scheduled trains",
    "Number of cancelled trains",
    "Number of trains delayed at departure",
    "Number of trains delayed at arrival",
    "Average delay of late trains at departure",
    "Average delay of all trains at departure",
    "Average delay of late trains at arrival",
    "Average delay of all trains at arrival",
    "Number of trains delayed > 15min",
    "Average delay of trains > 15min (if competing with flights)",
    "Number of trains delayed > 30min",
    "Number of trains delayed > 60min",
    "Pct delay due to external causes",
    "Pct delay due to infrastructure",
    "Pct delay due to traffic management",
    "Pct delay due to rolling stock",
    "Pct delay due to station management and equipment reuse",
    "Pct delay due to passenger handling (crowding, disabled persons, connections)",
]

# Convert to numeric where applicable
for col in positive_numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Identify rows with at least one negative value
negative_mask = df[positive_numeric_cols].lt(0).any(axis=1)
num_neg_rows = negative_mask.sum()
pct_neg_rows = (num_neg_rows / len(df)) * 100

print(f"⚠️ Rows with at least one negative value: {num_neg_rows} ({pct_neg_rows:.2f}%)")

# Optional preview of problematic rows
df_negative = df[negative_mask]
df_negative.head()

In [ ]:
# 🧹 Code Block 8 — Drop Rows with Negative Values

# Drop rows where any of the positive numeric columns have a negative value
initial_shape = df.shape
df = df[~negative_mask].copy()
final_shape = df.shape

# Replace impossible year values
year_clean = [["2026", "2022"], ["2028", "2022"], ["2069", "2019"], ["2121", "2021"]]

# Cleaning date using reference lists
for wrong_year, correct_year in year_clean:
    mask = df["Date"].dt.year == int(wrong_year)
    df.loc[mask, "Date"] = df.loc[mask, "Date"].apply(
        lambda d: d.replace(year=int(correct_year))
    )

print(f"✅ Dropped {initial_shape[0] - final_shape[0]} rows with negative values.")
print(f"📊 Remaining rows: {final_shape[0]}")

In [ ]:
# 🧹 Code Block 9 — Convert Columns to Integers

import pandas as pd

print(f"Initial shape: {df.shape}")
df.head()

# Convert all columns starting with 'Number of ' to integers (round if float)
for col in df.columns:
    if col.startswith("Number of "):
        df[col] = pd.to_numeric(df[col], errors="coerce").round().astype("Int64")

Clean : nb_delayed and nb_cancelled when above nb_scheduled_trains

In [ ]:
# Supprimer les lignes incohérentes
df = df[
    (df["Number of trains delayed at departure"] <= df["Number of scheduled trains"])
    & (df["Number of cancelled trains"] <= df["Number of scheduled trains"])
]

print(f"✅ Lignes incohérentes supprimées. Nouvelle forme de la dataframe : {df.shape}")

## Step 4 — Handle Textual NA and Station Name Clean-up

Textual NA values like `'nan'`, `'NA'`, or numeric typos in station names can break downstream logic. We'll:
- Remove rows with any textual 'nan'
- Detect and correct station names containing digits
- Apply title case to station names

In [ ]:
# 🔎 Code Block — Check Station Names Containing Digits

station_cleaned_name = [
    ["LAUSANNE"],
    ["STRASBOURG"],
    ["ANNECY"],
    ["ARRAS"],
    ["BARCELONA"],
    ["BESANCON FRANCHE COMTE TGV"],
    ["BORDEAUX ST JEAN"],
    ["CHAMBERY CHALLES LES EAUX"],
    ["DIJON VILLE"],
    ["DOUAI"],
    ["FRANCFORT"],
    ["GRENOBLE"],
    ["LYON PART DIEU"],
    ["MARNE LA VALLEE"],
    ["MARSEILLE ST CHARLES"],
    ["METZ"],
    ["MONTPELLIER"],
    ["NANCY"],
    ["NIMES"],
    ["PARIS MONTPARNASSE"],
    ["PARIS LYON"],
    ["PARIS EST"],
    ["PARIS NORD"],
    ["QUIMPER"],
    ["ST MALO"],
    ["REIMS"],
    ["RENNES"],
    ["STUTTGART"],
    ["TOULOUSE MATABIAU"],
    ["TOURCOING"],
    ["VALENCE ALIXAN TGV"],
    ["TOURS"],
    ["ZURICH"],
]
station_wrong_name = [
    ["L3USANNE"],
    ["SyRASBOURG"],
    ["ANNECq", "ANNECJ"],
    ["AqRAS", "ARRdS"],
    ["BARCELOBA"],
    ["BESANCOj FRANCHE COMTE TGV"],
    ["BORDEAUX ST JEAm", "BORDEAUX ST JwAN"],
    ["CHAMBQRY CHALLES LES EAUX"],
    ["DIJON VIPLE"],
    ["POUAI"],
    ["FRANCAORT"],
    ["GRENeBLE", "GRENOBLn"],
    [
        "LYOE PART DIEU",
        "LYON iART DIEU",
        "LYON PART DHEU",
        "LYON PART DIMU",
        "LYON PART DIYU",
    ],
    ["MARNE BA VALLEE", "MARNE LA VALLEK"],
    [
        "MARSEILLE eT CHARLES",
        "MARSEILLE ST CHARLQS",
        "MARSEILLE ST uHARLES",
        "MARSEIfLE ST CHARLES",
    ],
    ["MTTZ"],
    ["MONWPELLIER"],
    ["NANCB"],
    ["NIMEG"],
    [
        "PACIS MONTPARNASSE",
        "PARBS MONTPARNASSE",
        "PARES MONTPARNASSE",
        "PARIP MONTPARNASSE",
        "PARIS MOATPARNASSE",
        "PARIS MONIPARNASSE",
        "PARIS MONTPAONASSE",
        "PARIS MONTPARfASSE",
        "PARIS MONTPARNASSA",
        "PARIS MONTPARNAUSE",
        "PARIS MONTPTRNASSE",
    ],
    [
        "PAfIS LYON",
        "PALIS LYON",
        "PARIO LYON",
        "PARIS LAON",
        "PARIS LhON",
        "PARIS LvON",
        "PARIS LYaN",
        "PARIS LYOM",
        "PARIS LYOv",
        "PARIS UYON",
        "PARISgLYON",
        "PARISWLYON",
        "PARrS LYON",
        "PAVIS LYON",
        "HARIS LYON",
        "CARIS LYON",
        "PARIS LYOA",
        "PARIS CYON",
        "PARIS QYON",
    ],
    ["PARIS EBT", "PARIS EdT"],
    ["PARIS IORD", "PARIS rORD", "PARIS XORD", "PARIS RORD"],
    ["QUIMPEK"],
    ["ST MALQ", "STXMALO"],
    ["UEIMS", "RRIMS"],
    ["RENNEN", "RENAES", "RENPES"],
    ["STUkTGART", "STUTTGAoT"],
    ["TOULOUSEKMATABIAU", "TOUUOUSE MATABIAU"],
    ["TOURAOING"],
    ["VAdENCE ALIXAN TGV", "VALENCE ALIQAN TGV", "VALEOCE ALIXAN TGV"],
    ["TOuRS"],
    ["ZARICH", "ZURICS", "ZURRCH", "ZUUICH"],
]

# Cleaning station names using reference lists
for i in range(len(station_wrong_name)):
    wrong_names = station_wrong_name[i]
    correct_name = station_cleaned_name[i][0]
    for i in range(len(wrong_names)):
        df["Departure station"] = df["Departure station"].replace(
            wrong_names[i], correct_name
        )
        df["Arrival station"] = df["Arrival station"].str.replace(
            wrong_names[i], correct_name, case=False
        )

print("✅ Station names cleaned successfully")
print(df["Departure station"].drop_duplicates())

# Check if any station name isn't valid (across both departure and arrival stations)
station_columns = ["Departure station", "Arrival station"]

# Create a mask for rows where either departure or arrival station contains a digit
contains_digit_mask = (
    df[station_columns].apply(lambda x: x.str.contains(r"\d", na=False)).any(axis=1)
)

# Count how many rows contain digits in station names
num_rows_with_digits = contains_digit_mask.sum()
print(f"🚨 Number of rows with station names containing digits: {num_rows_with_digits}")

# Optional: Preview problematic rows
df_with_digits = df[contains_digit_mask]
df_with_digits[["Departure station", "Arrival station"]].head()

In [ ]:
# 🧹 Code Block — Remove Rows with Station Names Containing Digits

# Remove rows where either departure or arrival station contains a digit
df_cleaned = df[~contains_digit_mask].copy()

# Check the number of rows removed
print(f"✅ Removed {num_rows_with_digits} rows where station names contained digits.")
print(f"📊 Remaining rows: {df_cleaned.shape[0]}")

In [ ]:
# 🗑️ Code Block — Drop Only Problematic Columns

# List of problematic columns to drop (using original column names)
columns_to_drop = [
    "Cancellation comments",
    "Departure delay comments",
    "Arrival delay comments",
]

# Drop specified columns
df_cleaned = df_cleaned.drop(columns=columns_to_drop)

# Check the new number of columns
print(f"✅ Dropped specified columns.")
print(f"📊 Remaining columns: {df_cleaned.shape[1]}")

In [ ]:
# 🧹 Code Block — Drop NaN Values and Inspect Remaining Data

# Drop rows with any NaN values in the dataset
df_cleaned = df_cleaned.dropna()

# Check the new number of rows after dropping NaN values
print(f"✅ Rows after dropping NaN values: {df_cleaned.shape[0]}")

# Optional: Check for columns still containing NaN values (should be zero)
nan_columns = df_cleaned.isna().sum()
print(f"❌ Columns with NaN values after drop: {nan_columns[nan_columns > 0]}")

# Check for any remaining duplicates in the data
duplicates = df_cleaned.duplicated().sum()
print(f"⚠️ Number of duplicate rows: {duplicates}")

## Step 5 — Delay Consistency Checks

We need to ensure that when delays are reported, the corresponding average delay is greater than zero.
This checks for rows where delays are indicated but the average delay for those trains is incorrectly reported as zero or negative.

In [ ]:
# 🚦 Define inconsistency masks using original column names:
dep_bad = (df_cleaned["Number of trains delayed at departure"] > 0) & (
    df_cleaned["Average delay of all trains at departure"] <= 0
)

arr_bad = (df_cleaned["Number of trains delayed at arrival"] > 0) & (
    df_cleaned["Average delay of all trains at arrival"] <= 0
)

# 🧹 Combine both conditions
mask_inconsistent = dep_bad | arr_bad

# 🗑️ Drop inconsistent rows
print(f"⚠️ Inconsistent delay rows to drop: {mask_inconsistent.sum()}")
df_cleaned = df_cleaned[~mask_inconsistent].copy()
print(f"✅ Remaining rows after consistency check: {df_cleaned.shape[0]}")

## Step 6 — Feature Engineering

Here we create useful features like `year`, `month` from the date and `departure_delay_ratio`, which helps for downstream analysis.

In [ ]:
# 🚀 Year & month extraction
df_cleaned["year"] = df_cleaned["Date"].dt.year
df_cleaned["month"] = df_cleaned["Date"].dt.month

# 🚦 Delay ratio calculation (using original column names)
df_cleaned["departure_delay_ratio"] = (
    df_cleaned["Number of trains delayed at departure"]
    / df_cleaned["Number of scheduled trains"]
)

print("✅ Added features: year, month, departure_delay_ratio")
df_cleaned.head(3)

## Step 7 — Export Cleaned Dataset

Finally, save the cleaned dataset for further analysis or model building.

## Sub-Step 7.1

Changes to help us pass the tests

In [ ]:
df_cleaned.rename(columns={"date": "Date"}, inplace=True)

df_cleaned["Date"] = df_cleaned["Date"].dt.strftime("%Y-%m")

In [ ]:
df_cleaned.to_csv("cleaned_dataset.csv", index=False)
print("✅ Dataset exported as cleaned_dataset.csv")

## 📊 Step 2: Data Visualization & Analysis

Now that we’ve cleaned the dataset (`df_cleaned`), we can begin exploring the data to understand the nature of train delays. This step aims to uncover patterns and insights that will guide our future predictive modeling efforts.

We will:
- Generate summary statistics to understand the structure and scale of our data.
- Visualize the distribution of delays.
- Identify the most impacted stations.
- Examine

#### 2.1 Summary Statistics

We begin by inspecting descriptive statistics for numeric features. This provides an overview of central tendencies, dispersion, and potential outliers in delay-related data.

In [ ]:
# Basic info and summary statistics
df_cleaned.describe(include="all")
df_cleaned.info()

#### 2.2 Delay Distributions

We examine the distribution of delay durations to identify typical delay lengths and detect skewed distributions or outliers.

In [ ]:
# 📊 Import required libraries
import seaborn as sns
import matplotlib.pyplot as plt

# 🔶 Distribution of average delay of late trains
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot for average delay of late trains at departure
sns.histplot(
    df_cleaned["Average delay of late trains at departure"],
    bins=30,
    ax=axes[0],
    kde=True,
)
axes[0].set_title("Avg Delay of Late Trains at Departure")

# Plot for average delay of late trains at arrival
sns.histplot(
    df_cleaned["Average delay of late trains at arrival"], bins=30, ax=axes[1], kde=True
)
axes[1].set_title("Avg Delay of Late Trains at Arrival")

# Tight layout for better spacing and visualization
plt.tight_layout()
plt.show()

#### 2.3 Delays by Station

We analyze how delays vary across different departure and arrival stations. This helps identify problematic locations.

In [ ]:
# 📊 Top departure stations with the most delayed trains
top_depart = (
    df_cleaned.groupby("Departure station")["Number of trains delayed at departure"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

# Plotting the results
plt.figure(figsize=(10, 5))
sns.barplot(x=top_depart.values, y=top_depart.index)
plt.title("Top 10 Departure Stations by Total Delayed Trains")
plt.xlabel("Number of Delayed Trains")
plt.show()

### 📆 2.4 Delays by Time of Year

Understanding how delays evolve over time (e.g., seasonal trends, disruptions) is key for operational forecasting.


In [ ]:
# 📅 Convert to datetime and filter up to current year (2025)
df_cleaned["Date"] = pd.to_datetime(df_cleaned["Date"], errors="coerce")
df_cleaned = df_cleaned[df_cleaned["Date"].dt.year <= 2025]
df_cleaned["year_month"] = df_cleaned["Date"].dt.to_period("M").dt.to_timestamp()

# 📊 Aggregate and plot
monthly_delays = df_cleaned.groupby("year_month")[
    "Number of trains delayed at departure"
].sum()

# Plotting the monthly delays
plt.figure(figsize=(12, 5))
monthly_delays.plot(marker="o")
plt.title("Monthly Number of Trains Delayed at Departure")
plt.ylabel("Number of Delays")
plt.xlabel("Month")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

### 🔗 2.5 Correlation Between Delay Features

We’ll use a heatmap to explore correlations between different delay-related features. This helps us understand how factors interact.


In [ ]:
# Select numeric columns related to delay
delay_cols = df_cleaned.filter(like="delay").select_dtypes(include="number")

plt.figure(figsize=(12, 8))
sns.heatmap(delay_cols.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Between Delay-Related Features")
plt.show()